# Annexe B — Cahier de code, Chapitre 7
## Couleur et photométrie

Ce notebook accompagne le chapitre 7 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Image couleur d'exemple
import numpy as np
from skimage import data
import matplotlib.pyplot as plt

img = data.astronaut()                   # uint8, H×W×3, RGB
plt.imshow(img); plt.title("image (RGB)"); plt.show()

## 7.1 — Luminance (RGB → gris)

In [ ]:
from skimage.color import rgb2gray
gris = rgb2gray(img)       # moyenne pondérée perceptuelle 0.21R+0.72G+0.07B

## 7.2 — RGB → HSV

In [ ]:
from skimage.color import rgb2hsv
hsv = rgb2hsv(img)         # teinte / saturation / valeur
teinte = hsv[..., 0]

## 7.3 — CIELAB et la mesure perceptuelle

In [ ]:
# distance de couleur perceptuelle (ΔE) dans l'espace Lab
from skimage.color import rgb2lab, deltaE_ciede2000
lab = rgb2lab(img)
dE = deltaE_ciede2000(lab[:10, :10], lab[10:20, :10])

## 7.4 — Gamut et conversion RGB → CMYK

In [ ]:
# conversion naïve (sans profil ICC)
rgb = img / 255.0
k = 1 - rgb.max(axis=2)
c = (1 - rgb[..., 0] - k) / (1 - k + 1e-9)
m = (1 - rgb[..., 1] - k) / (1 - k + 1e-9)
y = (1 - rgb[..., 2] - k) / (1 - k + 1e-9)

## 7.5 — Égalisation d'histogramme et CLAHE

In [ ]:
from skimage.exposure import equalize_hist, equalize_adapthist
from skimage.color import rgb2gray
g = rgb2gray(img)
eg = equalize_hist(g)              # global
clahe = equalize_adapthist(g, clip_limit=0.03)   # local, à contraste limité

## 7.6 — Correction gamma et balance des blancs

In [ ]:
from skimage.exposure import adjust_gamma
gamma = adjust_gamma(img, gamma=0.5)
# balance des blancs « monde gris » : chaque canal ramené à la même moyenne
moy = img.reshape(-1, 3).mean(axis=0)
wb = np.clip(img * (moy.mean() / moy), 0, 255).astype(np.uint8)